# Clean and load `previous_application_extended`

**Aroa's addition.** While digging into Q3 (why do returning clients
default more than new ones?), the trimmed `previous_application` table
Carla built wasn't enough to explain a residual gap — it doesn't have
loan amounts or dates. This notebook goes back to the raw
`previous_application.csv` for just those extra columns
(`AMT_CREDIT`, `AMT_ANNUITY`, `DAYS_DECISION`), cleans them, and loads
them into a **separate** table, `previous_application_extended` — it
does not replace or modify Carla's `previous_application` table.

See `sql_scripts/q3_extended_repeat_client_drivers.sql` for the
analysis this feeds into.

In [1]:
import pandas as pd
import yaml
import getpass
from urllib.parse import quote_plus
from sqlalchemy import create_engine

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

## Load

In [2]:
previous_extended = pd.read_csv(
    config['input_data']['previous_application'],
    usecols=['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_STATUS', 'AMT_CREDIT', 'AMT_ANNUITY', 'DAYS_DECISION'],
)
previous_extended.shape

(1670214, 6)

## Explore

In [3]:
previous_extended.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 6 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   SK_ID_PREV            1670214 non-null  int64  
 1   SK_ID_CURR            1670214 non-null  int64  
 2   NAME_CONTRACT_STATUS  1670214 non-null  object 
 3   AMT_CREDIT            1670213 non-null  float64
 4   AMT_ANNUITY           1297979 non-null  float64
 5   DAYS_DECISION         1670214 non-null  int64  
dtypes: float64(2), int64(3), object(1)
memory usage: 76.4+ MB


In [4]:
previous_extended.isna().sum()

SK_ID_PREV                   0
SK_ID_CURR                   0
NAME_CONTRACT_STATUS         0
AMT_CREDIT                   1
AMT_ANNUITY             372235
DAYS_DECISION                 0
dtype: int64

In [5]:
previous_extended["SK_ID_PREV"].duplicated().sum()

np.int64(0)

## Clean

- `AMT_CREDIT`: 1 null row -- negligible, drop.
- `AMT_ANNUITY`: nulls concentrated in `Refused`/`Canceled` rows (no
  annuity was ever set because the loan never went through) -- fill
  with 0 rather than drop, since those rows still carry real
  `NAME_CONTRACT_STATUS`/`DAYS_DECISION` information the extended
  analysis needs.
- `DAYS_DECISION`: no nulls, no cleaning needed -- it's a negative
  offset in days from the current application, already numeric.

In [6]:
previous_extended = previous_extended.dropna(subset=["AMT_CREDIT"])
previous_extended["AMT_ANNUITY"] = previous_extended["AMT_ANNUITY"].fillna(0)
previous_extended.shape

(1670213, 6)

In [7]:
# Verify: nulls should be gone
previous_extended.isna().sum()

SK_ID_PREV              0
SK_ID_CURR              0
NAME_CONTRACT_STATUS    0
AMT_CREDIT              0
AMT_ANNUITY             0
DAYS_DECISION           0
dtype: int64

## Save clean CSV

In [8]:
previous_extended.to_csv(config['output_data']['previous_application_extended'], index=False)

## Load into MySQL

In [9]:
db = config['database']
password = getpass.getpass("MySQL password (press Enter if none): ")
engine = create_engine(f"mysql+pymysql://{db['user']}:{quote_plus(password)}@{db['host']}/{db['name']}")

# Same FK-safety filter used for the other tables: previous_application.csv
# includes applicants outside application_train.csv (Kaggle's test set),
# which would violate the foreign key to application.
existing_ids = pd.read_sql("SELECT SK_ID_CURR FROM application", engine)["SK_ID_CURR"]
previous_extended = previous_extended[previous_extended["SK_ID_CURR"].isin(existing_ids)]
previous_extended.shape

(1413552, 6)

In [10]:
previous_extended.to_sql(
    "previous_application_extended",
    engine,
    if_exists="append",
    index=False,
    chunksize=10000,
    method="multi",
)

1413552